# SQL Business Metrics Query Design

This notebook demonstrates designing reusable SQL business metrics:
1. **Monthly Active Users (MAU)** with `CASE WHEN` conditional aggregation.
2. **Revenue by Customer Segment** computing 5 metrics per segment.
3. **Conversion Funnel Analytics** calculating daily signup-to-purchase conversion rates.
4. **Calling Version-Controlled `.sql` Files from Python** (`pd.read_sql`).
5. **Validating Metric Integrity & Business Assertions**.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

db_path = '../analytics.db'
if not os.path.exists(db_path):
    db_path = 'analytics.db'

engine = create_engine(f'sqlite:///{db_path}')
print(f"Connected to database: {db_path}")

## Task 1: Monthly Active Users (MAU) Metric

In [2]:
def load_query(query_name):
    path = f'../queries/{query_name}.sql'
    if not os.path.exists(path):
        path = f'queries/{query_name}.sql'
    with open(path, 'r') as f:
        return f.read()

mau_sql = load_query('monthly_active_users')
mau_df = pd.read_sql(mau_sql, engine)
print("Monthly Active Users (MAU) Metric:")
print(mau_df.head())

## Task 2: Revenue per Segment Metric

In [3]:
revenue_sql = load_query('revenue_by_segment')
revenue_df = pd.read_sql(revenue_sql, engine)
print("Revenue by Segment Metric:")
print(revenue_df.head(6))

## Task 3: Conversion Funnel Analytics

In [4]:
funnel_sql = load_query('conversion_funnel')
funnel_df = pd.read_sql(funnel_sql, engine)
print("Conversion Funnel Metric:")
print(funnel_df.head())

## Task 5: Metric Validation Assertions

In [5]:
assert mau_df.isnull().sum().sum() == 0, "MAU has nulls"
assert revenue_df.isnull().sum().sum() == 0, "Revenue has nulls"
assert funnel_df.isnull().sum().sum() == 0, "Funnel has nulls"
assert (revenue_df['monthly_revenue'] > 0).all(), "Revenue <= 0"
assert (funnel_df['conversion_pct'] >= 0).all() and (funnel_df['conversion_pct'] <= 100).all(), "Conversion out of range"
print("[OK] All SQL metric query results validated successfully!")